In [12]:
# Cell 1: Environment Setup & Latest Blender Installation
import sys
import os

print(f"Python version: {sys.version}")

# 1. Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# 2. Define Project Workspace on Drive
DRIVE_ROOT = "/content/drive/MyDrive/Blender_Production"
WORKSPACE = {
    "assets": os.path.join(DRIVE_ROOT, "assets"),
    "renders": os.path.join(DRIVE_ROOT, "renders"),
    "logs": os.path.join(DRIVE_ROOT, "logs"),
    "recovery": os.path.join(DRIVE_ROOT, "recovery"),
    "exports": os.path.join(DRIVE_ROOT, "exports")
}

for path in WORKSPACE.values():
    os.makedirs(path, exist_ok=True)
print(f"✅ Workspace created at: {DRIVE_ROOT}")

# 3. Install Blender 5.2 LTS (bpy) for Python 3.13
# The extra-index-url points to Blender's official wheel repository.
!pip install bpy==5.2.3 --extra-index-url https://download.blender.org/pypi/

import bpy
print(f"✅ bpy imported successfully. Blender version: {bpy.app.version_string}")

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Mounted at /content/drive
✅ Workspace created at: /content/drive/MyDrive/Blender_Production
Looking in indexes: https://pypi.org/simple, https://download.blender.org/pypi/
✅ bpy imported successfully. Blender version: 5.2.3 LTS


In [13]:
# Cell 2: Recovery & Logging Configuration
import bpy
import sys

# 1. Configure Auto-Save to Drive
# This tells Blender where to store its automatic backups.
bpy.context.preferences.filepaths.temporary_directory = WORKSPACE["recovery"]
bpy.context.preferences.filepaths.use_auto_save_temporary_files = True
bpy.context.preferences.filepaths.auto_save_time = 2  # Save every 2 minutes
print(f"✅ Auto-Save enabled. Backups will be stored in: {WORKSPACE['recovery']}")

# 2. Redirect Blender's Output to a Log File
log_file_path = os.path.join(WORKSPACE["logs"], "blender_session.log")
log_file = open(log_file_path, "a")  # Append mode
sys.stdout = log_file
sys.stderr = log_file

print("--- New Blender Session Started ---")
print(f"Workspace: {WORKSPACE}")
print("All print statements and errors from this point forward will be logged here.")

In [14]:
# Cell 3: Procedural Scene Builder
import bpy
import math

# 1. Clear Default Scene
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)

# 2. Geometry
# High-poly sphere for smooth reflections
bpy.ops.mesh.primitive_uv_sphere_add(segments=128, ring_count=64, radius=2.0)
subject = bpy.context.active_object
bpy.ops.object.shade_smooth()

# Floor plane
bpy.ops.mesh.primitive_plane_add(size=20, location=(0, 0, -2))
floor = bpy.context.active_object

# 3. Materials (Blender 5.2 Principled BSDF)
# Subject: Red Metallic
mat_subject = bpy.data.materials.new(name="Realistic_Metal")
bsdf = mat_subject.node_tree.nodes["Principled BSDF"]
bsdf.inputs["Base Color"].default_value = (0.8, 0.05, 0.1, 1.0)
bsdf.inputs["Metallic"].default_value = 1.0
bsdf.inputs["Roughness"].default_value = 0.2
subject.data.materials.append(mat_subject)

# Floor: Dark Glossy
mat_floor = bpy.data.materials.new(name="Studio_Floor")
bsdf_floor = mat_floor.node_tree.nodes["Principled BSDF"]
bsdf_floor.inputs["Base Color"].default_value = (0.02, 0.02, 0.02, 1.0)
bsdf_floor.inputs["Roughness"].default_value = 0.1
floor.data.materials.append(mat_floor)

# 4. Camera with Depth of Field
bpy.ops.object.camera_add(location=(7, -7, 4))
camera = bpy.context.active_object
bpy.context.scene.camera = camera
track_to = camera.constraints.new(type='TRACK_TO')
track_to.target = subject
track_to.track_axis = 'TRACK_NEGATIVE_Z'
track_to.up_axis = 'UP_Y'
camera.data.dof.use_dof = True
camera.data.dof.focus_object = subject

# 5. Three-Point Lighting
# Key Light
bpy.ops.object.light_add(type='AREA', location=(5, -5, 6))
key_light = bpy.context.active_object
key_light.data.energy = 2000
key_light.data.size = 3.0

# Fill Light
bpy.ops.object.light_add(type='AREA', location=(-5, -3, 3))
fill_light = bpy.context.active_object
fill_light.data.energy = 500
fill_light.data.size = 5.0

# Rim Light
bpy.ops.object.light_add(type='AREA', location=(0, 6, 5))
rim_light = bpy.context.active_object
rim_light.data.energy = 3000
rim_light.data.size = 2.0

print("✅ Scene built successfully.")

In [15]:
# Cell 4: Asset Import/Export Functions
import bpy

def import_asset(filepath):
    """Imports a 3D asset based on its file extension."""
    ext = filepath.lower().split('.')[-1]
    if ext in ['glb', 'gltf']:
        bpy.ops.import_scene.gltf(filepath=filepath)
    elif ext == 'fbx':
        bpy.ops.import_scene.fbx(filepath=filepath)
    elif ext == 'obj':
        bpy.ops.wm.obj_import(filepath=filepath)
    elif ext == 'usd' or ext == 'usda':
        bpy.ops.import_scene.usd(filepath=filepath)
    else:
        print(f"⚠️ Unsupported format: {ext}")
        return
    print(f"✅ Imported: {filepath}")

def export_scene(filepath, format='GLB'):
    """Exports the current scene in the specified format."""
    if format == 'GLB':
        bpy.ops.export_scene.gltf(filepath=filepath, export_format='GLB')
    elif format == 'FBX':
        bpy.ops.export_scene.fbx(filepath=filepath)
    elif format == 'OBJ':
        bpy.ops.wm.obj_export(filepath=filepath)
    elif format == 'USD':
        bpy.ops.export_scene.usd(filepath=filepath)
    print(f"✅ Exported: {filepath}")

# Example: Export the current scene as a GLB file
# export_scene(os.path.join(WORKSPACE["exports"], "scene.glb"), format='GLB')

In [16]:
# Cell 5: Lip-Sync Viseme Shape Keys
import bpy

def create_viseme_keys(mesh_obj):
    """Creates the five standard VRM visemes on the specified mesh."""
    if not mesh_obj.data.shape_keys:
        bpy.context.view_layer.objects.active = mesh_obj
        bpy.ops.object.shape_key_add(from_mix=False)  # Create Basis key

    visemes = ["A", "I", "U", "E", "O"]
    for name in visemes:
        if name not in mesh_obj.data.shape_keys.key_blocks:
            key = mesh_obj.shape_key_add(name=name, from_mix=False)
            key.value = 0.0  # Initialize to zero
    print(f"✅ Viseme keys created on {mesh_obj.name}: {visemes}")

# Example: Apply to a mesh named 'Character'
# character = bpy.data.objects.get("Character")
# if character:
#     create_viseme_keys(character)

In [17]:
# Cell 6: Audio Visualization (Sound to F-Curves)
import bpy

def bake_sound_to_fcurve(obj_name, audio_path):
    """Bakes an audio file to the selected object's F-Curves."""
    obj = bpy.data.objects.get(obj_name)
    if not obj:
        print(f"⚠️ Object '{obj_name}' not found.")
        return

    # Ensure the object is selected and active
    bpy.context.view_layer.objects.active = obj
    obj.select_set(True)

    # Switch to the Graph Editor to use the operator
    # (In headless mode, we can call the operator directly if context is set)
    try:
        bpy.ops.graph.sound_bake(filepath=audio_path)
        print(f"✅ Sound baked to F-Curves for {obj_name}")
    except Exception as e:
        print(f"⚠️ Sound bake failed: {e}")

# Example: Bake an audio file to the 'subject' sphere
# audio_file = os.path.join(WORKSPACE["assets"], "beat.wav")
# if os.path.exists(audio_file):
#     bake_sound_to_fcurve("subject", audio_file)

In [19]:
# Cell 8: Keep-Alive Instructions (Run in Browser Console)
print("""
To help prevent Colab from disconnecting due to inactivity:
1. On this Colab notebook page, press F12 (or Ctrl+Shift+I) to open Developer Tools.
2. Go to the 'Console' tab.
3. Paste the following code and press Enter:

function ClickConnect(){
  console.log("Keeping Colab alive...");
  document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(ClickConnect, 60000);

This will click the 'Connect' button every 60 seconds.
""")

In [22]:
# Cell 8: Quality Control - Input vs. Output Comparison
import bpy
import os
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

# UI Layout for input paths (defaults match Cell 7)
input_img_path = widgets.Text(
    value="/content/drive/MyDrive/Blender_Production/assets/reference.jpg",
    description='2D Input:', layout=widgets.Layout(width='80%')
)
output_img_path = widgets.Text(
    value="/content/drive/MyDrive/Blender_Production/renders/final_asset_preview.png",
    description='3D Render:', layout=widgets.Layout(width='80%')
)

btn_compare = widgets.Button(description="⚖️ Compare Results", button_style='primary')
out_compare = widgets.Output()

def on_compare(b):
    with out_compare:
        clear_output(wait=True)

        # 1. Display Visual Comparison
        print("👁️ VISUAL COMPARISON")
        print("Left: Original 2D Intent | Right: Generated 3D Output\n")

        img_left = None
        img_right = None

        if os.path.exists(input_img_path.value):
            img_left = widgets.Image(value=open(input_img_path.value, 'rb').read(), format='jpg', width=400)
        else:
            img_left = widgets.Label("❌ Input image not found.")

        if os.path.exists(output_img_path.value):
            img_right = widgets.Image(value=open(output_img_path.value, 'rb').read(), format='png', width=400)
        else:
            img_right = widgets.Label("❌ Output render not found. Did you render in Cell 7?")

        display(widgets.HBox([img_left, img_right]))

        # 2. Objective Mathematical Analysis
        print("\n" + "="*50)
        print("📊 OBJECTIVE MESH DATA (Is it a real 3D asset?)")
        print("="*50)

        # Find the most recently generated 3D object
        target_obj = None
        for obj in bpy.data.objects:
            if obj.type == 'MESH' and obj.name not in ['Camera', 'Light', 'Cube']:
                target_obj = obj
                break

        if target_obj:
            vert_count = len(target_obj.data.vertices)
            poly_count = len(target_obj.data.polygons)
            dims = target_obj.dimensions

            print(f"Asset Name:  {target_obj.name}")
            print(f"Vertices:    {vert_count:,} (Nano-precise geometry)")
            print(f"Polygons:    {poly_count:,}")
            print(f"Dimensions:  X: {dims.x:.3f}m, Y: {dims.y:.3f}m, Z: {dims.z:.3f}m")
            print(f"Materials:   {len(target_obj.data.materials)}")

            if vert_count > 10:
                print("\n✅ Validation Passed: This is a fully realized 3D mesh.")
            else:
                print("\n⚠️ Warning: Mesh complexity is very low.")
        else:
            print("No 3D mesh found in the current scene.")

        print("="*50)
        print("🤔 SUBJECTIVE EVALUATION: Does the 3D output match your 2D intent?")
        print("If yes, proceed to the final export cell.")

btn_compare.on_click(on_compare)

# Display Dashboard
print("🎛️ INTENTION VALIDATION DASHBOARD")
display(widgets.VBox([
    input_img_path,
    output_img_path,
    btn_compare,
    out_compare
]))

In [23]:
# Cell 7: Professional Studio with Real-Time Process Tracking
import bpy
import bmesh
import math
import os
import subprocess
import sys
import time
import ipywidgets as widgets
from IPython.display import display, Image as IPImage, clear_output

# ==========================================
# 1. THE UNIVERSAL PROCESS TRACKER
# ==========================================
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', bar_style='info')
console_out = widgets.Output(layout={'border': '1px solid #444', 'height': '150px', 'overflow_y': 'auto'})
start_time = [0.0]

def update_tracker(percent, message, style='info'):
    """Updates both the progress bar and the live console log."""
    progress_bar.value = percent
    progress_bar.bar_style = style
    elapsed = time.time() - start_time[0]
    with console_out:
        print(f"[{elapsed:05.1f}s] {percent:3d}% | {message}")

def reset_tracker():
    start_time[0] = time.time()
    update_tracker(0, "Initializing process...")
    progress_bar.description = 'Progress:'

# ==========================================
# 2. SETUP BASE SCENE
# ==========================================
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)
bpy.ops.object.camera_add(location=(0, -10, 6), rotation=(math.radians(55), 0, 0))
bpy.context.scene.camera = bpy.context.object
bpy.ops.object.light_add(type='AREA', location=(5, -5, 8))
bpy.context.object.data.energy = 1500

# ==========================================
# 3. ENGINE A: NATIVE NANO-RELIEF (Tracked)
# ==========================================
def native_relief_sculpt(image_path, subdivisions, depth):
    reset_tracker()

    update_tracker(10, f"Generating {subdivisions}x{subdivisions} mesh grid...")
    mesh = bpy.data.meshes.new("Nano_Relief_Mesh")
    obj = bpy.data.objects.new("Nano_Relief", mesh)
    bpy.context.collection.objects.link(obj)

    bm = bmesh.new()
    bmesh.ops.create_grid(bm, x_segments=subdivisions, y_segments=subdivisions, size=5.0)
    bm.to_mesh(mesh)
    bm.free()
    for poly in mesh.polygons: poly.use_smooth = True

    update_tracker(40, "Loading image heightmap...")
    tex = bpy.data.textures.new("Heightmap", type='IMAGE')
    tex.image = bpy.data.images.load(image_path)

    update_tracker(60, "Applying Displacement Modifier (Sculpting)...")
    mod = obj.modifiers.new("Displace", 'DISPLACE')
    mod.texture = tex
    mod.strength = depth
    mod.mid_level = 0.5

    update_tracker(85, "Applying Subdivision Surface...")
    mod_sub = obj.modifiers.new("Subsurf", 'SUBSURF')
    mod_sub.levels = 2
    mod_sub.render_levels = 3

    update_tracker(100, "✅ Native Sculpt Complete!", 'success')
    return obj

# ==========================================
# 4. ENGINE B: AI FULL 3D (TripoSR with Live Tracking)
# ==========================================
def ai_generate_3d(image_path):
    reset_tracker()
    repo_dir = "/content/TripoSR"

    # Step 1: Setup
    if not os.path.exists(repo_dir):
        update_tracker(5, "Downloading TripoSR AI Repository...")
        subprocess.run(["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git", repo_dir], capture_output=True)
        update_tracker(15, "Installing AI Dependencies...")
        subprocess.run(["pip", "install", "-q", "-r", f"{repo_dir}/requirements.txt"], capture_output=True)
        subprocess.run(["pip", "install", "-q", "torchmcubes", "rembg", "trimesh", "omegaconf"], capture_output=True)

    # Step 2: AI Generation (Streaming output in real-time)
    update_tracker(30, "Initializing PyTorch Engine & Loading Weights...")
    out_dir = "/content/drive/MyDrive/Blender_Production/assets/ai_output"
    os.makedirs(out_dir, exist_ok=True)

    cmd = [sys.executable, f"{repo_dir}/run.py", image_path, "--output-dir", out_dir, "--model-save-format", "obj"]

    update_tracker(50, "AI is generating full 3D geometry (this takes 1-2 mins)...")

    # Run process and stream logs to console
    process = subprocess.Popen(cmd, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if line.strip():
            with console_out:
                print(f"  [AI Engine] {line.strip()}")
    process.wait()

    update_tracker(85, "Importing generated OBJ into Blender...")
    obj_path = os.path.join(out_dir, "mesh.obj")
    if os.path.exists(obj_path):
        bpy.ops.wm.obj_import(filepath=obj_path)
        obj = bpy.context.selected_objects[0]
        obj.name = "AI_Generated_Asset"
        update_tracker(100, "✅ AI Generation Complete!", 'success')
        return obj

    update_tracker(100, "❌ AI Generation Failed.", 'danger')
    return None

# ==========================================
# 5. RENDER TRACKER (Hooks into Blender)
# ==========================================
def render_progress_handler(scene):
    # Blender fires this for every frame during animation render
    frame = scene.frame_current
    total = scene.frame_end - scene.frame_start
    if total > 0:
        percent = int(((frame - scene.frame_start) / total) * 100)
        update_tracker(percent, f"Rendering Frame {frame}/{scene.frame_end}")

# ==========================================
# 6. THE MASTER UI DASHBOARD
# ==========================================
img_input = widgets.Text(value="/content/drive/MyDrive/Blender_Production/assets/reference.jpg", description='2D Image:', layout=widgets.Layout(width='70%'))
engine_dropdown = widgets.Dropdown(options=['Native Nano-Relief', 'AI Full 3D (TripoSR)'], value='AI Full 3D (TripoSR)', description='Engine:')
quality_slider = widgets.IntSlider(value=128, min=32, max=512, description='Resolution:')
depth_slider = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, description='Depth:')

btn_process = widgets.Button(description="⚙️ Generate 3D Asset", button_style='primary')
btn_render = widgets.Button(description="📸 Render Preview", button_style='success')

# PROCESS BUTTON LOGIC
def on_process(b):
    with console_out:
        clear_output()
    if not os.path.exists(img_input.value):
        update_tracker(0, f"❌ Image not found at {img_input.value}", 'danger')
        return

    if engine_dropdown.value == 'Native Nano-Relief':
        native_relief_sculpt(img_input.value, quality_slider.value, depth_slider.value)
    else:
        ai_generate_3d(img_input.value)

# RENDER BUTTON LOGIC
def on_render(b):
    with console_out:
        clear_output()
    reset_tracker()
    update_tracker(10, "Configuring Cycles Render Engine...")

    scene = bpy.context.scene
    scene.render.engine = 'CYCLES'
    scene.cycles.samples = 64
    scene.render.resolution_x = 800
    scene.render.resolution_y = 800

    render_path = "/content/drive/MyDrive/Blender_Production/renders/final_asset_preview.png"
    scene.render.filepath = render_path

    update_tracker(30, "Rendering in progress (Cycles CPU)...")
    bpy.ops.render.render(write_still=True)

    update_tracker(100, "✅ Render Complete! Displaying image...", 'success')
    display(IPImage(filename=render_path))

btn_process.on_click(on_process)
btn_render.on_click(on_render)

# ==========================================
# 7. DISPLAY INTERFACE
# ==========================================
print("🎛️ PROFESSIONAL STUDIO WITH REAL-TIME TRACKING")
print("Monitor the exact performance and file output status below:\n")

display(widgets.VBox([
    img_input,
    widgets.HBox([engine_dropdown, quality_slider, depth_slider]),
    widgets.HBox([btn_process, btn_render]),
    widgets.HTML("<hr><b>Process Monitor</b>"),
    progress_bar,
    console_out
]))